# 22 · Load: BLS Employment Projections, education requirements and national projections → `staging`

**Source card:** `bls_ep_education.md` (§6 issues, §7 staging tables). Explored in: notebook 04 (Tables 5.3, 5.4) and notebook 06 §6f (Table 1.2, decision D1).

| Table | One row = | Expected rows |
|---|---|---|
| `stg_bls__ep_entry_requirements` | job (national, 2025 edition) | 831 |
| `stg_bls__ep_attainment` | job × education level of current workers (long) | 832 × 7 = 5,824 |
| `stg_bls__ep_occupation_projections` | job, national 2025–35 (Table 1.2): growth and openings rates for the D1 fill | 1,112 |

All national. NYC relevance comes from the join on SOC in core.

In [1]:
import re
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.py").exists())
sys.path.insert(0, str(PROJECT / "src"))
from common import log
from staging import connect, raw_files, stage

pd.set_option("display.max_columns", 40, "display.max_colwidth", 70, "display.width", 220)
NB = "22-load-bls-education.ipynb"
SOC = r"\d\d-\d{4}"
DASH = "—"   # BLS uses an em dash for "not available" / "no OOH page"
conn = connect()
summary = []

EDU = pd.read_excel(raw_files("bls_ep_education").path.iloc[-1], sheet_name=None, header=1, dtype=str, keep_default_na=False)
EDU_FILE = raw_files("bls_ep_education").file.iloc[-1]
print("education.xlsx sheets:", list(EDU))

education.xlsx sheets: ['Index', 'Table 5.1', 'Table 5.2', 'Table 5.3', 'Table 5.4']


## 1. Ordered categories (issue 4)

BLS's own order, low → high. Every label in the file must be in these lists (asserted), so a renamed category stops the notebook.

In [2]:
EDUCATION = ["No formal educational credential", "High school diploma or equivalent", "Some college, no degree",
             "Postsecondary nondegree award", "Associate's degree", "Bachelor's degree", "Master's degree",
             "Doctoral or professional degree"]
EXPERIENCE = ["None", "Less than 5 years", "5 years or more"]
TRAINING = ["None", "Short-term on-the-job training", "Moderate-term on-the-job training",
            "Long-term on-the-job training", "Apprenticeship", "Internship/residency"]
ATTAINMENT = ["Less than high school diploma", "High school diploma or equivalent", "Some college, no degree",
              "Associate's degree", "Bachelor's degree", "Master's degree", "Doctoral or professional degree"]

def rank(s: pd.Series, order: list[str], name: str) -> pd.Series:
    unknown = set(s) - set(order)
    assert not unknown, f"{name}: labels not in BLS's list: {unknown}"
    return s.map({v: i + 1 for i, v in enumerate(order)}).astype("Int64")

## 2. Table 5.4: entry requirements (issues 1, 4, 5)

Header in row 2, notes below the data: keep rows whose code matches `NN-NNNN`.

In [3]:
t = EDU["Table 5.4"].copy()
t.columns = [c.strip() for c in t.columns]
t = t.apply(lambda s: s.str.strip())
keep = t.iloc[:, 1].str.fullmatch(SOC)
log("Table 5.4: keep rows with a SOC code (drop notes)", len(t), keep.sum())
t = t[keep].reset_index(drop=True)
t.columns = ["soc_title", "soc_code", "education", "experience", "training", "ooh"]
t["education_rank"] = rank(t.education, EDUCATION, "education")
t["experience_rank"] = rank(t.experience, EXPERIENCE, "experience")
t["training_rank"] = rank(t.training, TRAINING, "training")
print("OOH column values:", t.ooh.where(t.ooh == DASH, "(has page)").value_counts().to_dict())
t["has_ooh_page"] = t.ooh.ne(DASH)
t["edition"] = 2025
t["source_file"] = EDU_FILE
req = t[["edition", "soc_code", "soc_title", "education", "education_rank", "experience", "experience_rank",
         "training", "training_rank", "has_ooh_page", "source_file"]]
display(req.education.value_counts().to_frame().T)

# Tie-out (card §5): Software Developers = Bachelor's / None / None; Registered Nurses = Bachelor's
chk = req.set_index("soc_code")
assert tuple(chk.loc["15-1252", ["education", "experience", "training"]]) == ("Bachelor's degree", "None", "None")
assert chk.loc["29-1141", "education"] == "Bachelor's degree"
summary.append(stage(req, "stg_bls__ep_entry_requirements", ["soc_code"], expected_rows=831,
                     source_key="bls_ep_education", notebook=NB, conn=conn,
                     comment="BLS EP Table 5.4 (2025 edition): typical entry education, related experience, on-the-job training. National."))

  Table 5.4: keep rows with a SOC code (drop notes): 832 -> 831 rows (-1)
OOH column values: {'(has page)': 606, '—': 225}


education,High school diploma or equivalent,Bachelor's degree,No formal educational credential,Doctoral or professional degree,Postsecondary nondegree award,Associate's degree,Master's degree,"Some college, no degree"
count,326,177,109,73,51,48,40,7


  stg_bls__ep_entry_requirements: 831 rows x 11 cols, key ['soc_code'] unique | local data/interim/staging/stg_bls__ep_entry_requirements.parquet | DB: 831


## 3. Table 5.3: what education current workers hold (issues 1–3)

The `[1]` mark in a title means BLS borrowed the profile from a broader ACS code (`acs_inexact`). Reshaped long (one row per level) so "bachelor's or more" is a simple filtered sum. Rows sum to 100 ± 0.2 (asserted).

In [4]:
a = EDU["Table 5.3"].copy()
a.columns = [c.strip() for c in a.columns]
a = a.apply(lambda s: s.str.strip())
keep = a.iloc[:, 1].str.fullmatch(SOC)
log("Table 5.3: keep rows with a SOC code (drop notes)", len(a), keep.sum())
a = a[keep].rename(columns={a.columns[0]: "soc_title", a.columns[1]: "soc_code"}).reset_index(drop=True)
assert list(a.columns[2:]) == ATTAINMENT, list(a.columns[2:])
a["acs_inexact"] = a.soc_title.str.contains(r"\[1\]", regex=True)
a["soc_title"] = a.soc_title.str.replace(r"\s*\[1\]", "", regex=True)
print(f"acs_inexact (borrowed profile): {a.acs_inexact.sum()} jobs")
for c in ATTAINMENT:
    a[c] = pd.to_numeric(a[c], errors="raise")
rowsum = a[ATTAINMENT].sum(axis=1)
print(f"Row sums: {rowsum.min():.1f} - {rowsum.max():.1f}"); assert rowsum.between(99.8, 100.2).all()

att = a.melt(id_vars=["soc_code", "soc_title", "acs_inexact"], value_vars=ATTAINMENT,
             var_name="attainment_level", value_name="pct_of_workers")
att["attainment_rank"] = rank(att.attainment_level, ATTAINMENT, "attainment")
att["edition"] = 2025
att["attainment_years"] = "2023-24"
att["source_file"] = EDU_FILE
att = att[["edition", "attainment_years", "soc_code", "soc_title", "attainment_level", "attainment_rank",
           "pct_of_workers", "acs_inexact", "source_file"]].sort_values(["soc_code", "attainment_rank"]).reset_index(drop=True)
rn = att[(att.soc_code == "29-1141") & (att.attainment_level == "Associate's degree")].pct_of_workers.iloc[0]
print(f"Registered nurses holding an associate's: {rn}% (nb 04: 24.9%)"); assert rn == 24.9
summary.append(stage(att, "stg_bls__ep_attainment", ["soc_code", "attainment_level"], expected_rows=5_824,
                     source_key="bls_ep_education", notebook=NB, conn=conn,
                     comment="BLS EP Table 5.3: % of workers 25+ by highest education, ACS 2023-24. Long format. National."))

  Table 5.3: keep rows with a SOC code (drop notes): 835 -> 832 rows (-3)
acs_inexact (borrowed profile): 458 jobs
Row sums: 99.8 - 100.2
Registered nurses holding an associate's: 24.9% (nb 04: 24.9%)
  stg_bls__ep_attainment: 5,824 rows x 9 cols, key ['soc_code', 'attainment_level'] unique | local data/interim/staging/stg_bls__ep_attainment.parquet | DB: 5824


## 4. Table 1.2 (`occupation.xlsx`): national projections 2025–35

Needed by the D1 fill in core: national growth rate and openings per job split the 2022-32 NYC residual across the collapsed codes. One row per code, line items and summary rows both (`occupation_type`); employment in thousands as published. `—` (not available) appears in self-employed % and median wage: NULL, counted below. The two OOH columns (link text and HTML link) aren't staged.

In [5]:
of = raw_files("bls_ep_occupation").iloc[-1]
o = pd.read_excel(of.path, sheet_name="Table 1.2", header=1, dtype=str, keep_default_na=False)
o = o.apply(lambda s: s.str.strip())
keep = o.iloc[:, 1].str.fullmatch(SOC)
log("Table 1.2: keep rows with a SOC code (drop notes)", len(o), keep.sum())
o = o[keep].reset_index(drop=True)
o = o.iloc[:, :15]
o.columns = ["soc_title", "soc_code", "occupation_type", "employment_2025_k", "employment_2035_k", "pct_of_total_2025",
             "pct_of_total_2035", "employment_change_k", "employment_change_pct", "pct_self_employed_2025",
             "annual_openings_k", "median_wage_2025", "education", "experience", "training"]
print(o.occupation_type.value_counts().to_dict())
for c in o.columns[3:12]:
    na = o[c] == DASH
    if na.any():
        print(f"  {c}: '{DASH}' -> NULL: {na.sum()}")
    o[c] = pd.to_numeric(o[c].mask(na), errors="raise").astype("float64")
o["median_wage_2025"] = o.median_wage_2025.astype("Int64")
blank_req = o.occupation_type.eq("Line item") & o.education.eq("")
assert not blank_req.any(), "line items without an entry requirement"
o["edition"] = "2025-2035"
o["source_file"] = of.file
proj = o[["edition", "soc_code", "soc_title", "occupation_type", "employment_2025_k", "employment_2035_k",
          "employment_change_k", "employment_change_pct", "annual_openings_k", "pct_of_total_2025", "pct_of_total_2035",
          "pct_self_employed_2025", "median_wage_2025", "source_file"]]

# Tie-outs: the total row (card §5: 170,280.8k), line items = Table 5.4's 831 codes, same requirements
tot = proj.set_index("soc_code").loc["00-0000"]
print(f"Total employment 2025: {tot.employment_2025_k:,.1f}k"); assert tot.employment_2025_k == 170280.8
li = o[o.occupation_type == "Line item"].set_index("soc_code")
assert set(li.index) == set(req.soc_code), "Table 1.2 line items != Table 5.4 codes"
same = (li.loc[req.soc_code, ["education", "experience", "training"]].values == req[["education", "experience", "training"]].values).all()
print("Requirements in 1.2 identical to 5.4:", same); assert same

# D1 inputs: the SOC 2018 parts of the collapsed / missing NYC codes the decision names
# (15-1256, 15-1257, 13-2098, 13-1198, 15-2098; full part lists come from the hybrid file in core)
D1_PARTS = ["15-1252", "15-1253", "15-1254", "15-1255", "13-2051", "13-2054", "13-2099", "13-1082", "13-1199",
            "15-2051", "15-2099"]
have = proj.set_index("soc_code").reindex(D1_PARTS)[["soc_title", "employment_2025_k", "employment_change_pct", "annual_openings_k"]]
assert have.notna().all().all(), "a D1 part has no national rate"
display(have)
summary.append(stage(proj, "stg_bls__ep_occupation_projections", ["soc_code"], expected_rows=1_112,
                     source_key="bls_ep_occupation", notebook=NB, conn=conn,
                     comment="BLS EP Table 1.2: national employment 2025 and 2035 (thousands), change, annual openings. Used by the D1 fill."))

  Table 1.2: keep rows with a SOC code (drop notes): 1,117 -> 1,112 rows (-5)
{'Line item': 831, 'Summary': 281}
  pct_self_employed_2025: '—' -> NULL: 220
  median_wage_2025: '—' -> NULL: 9
Total employment 2025: 170,280.8k
Requirements in 1.2 identical to 5.4: True


,soc_title,employment_2025_k,employment_change_pct,annual_openings_k
soc_code,,,,
15-1252,Software developers,1717.8,10.2,95.3
15-1253,Software quality assurance analysts and testers,187.6,5.7,10.7
15-1254,Web developers,87.4,3.8,5.3
15-1255,Web and digital interface designers,132.7,6.0,8.3
13-2051,Financial and investment analysts,377.2,7.2,24.7
13-2054,Financial risk specialists,66.0,7.4,4.8
13-2099,"Financial specialists, all other",137.8,4.1,9.4
13-1082,Project management specialists,1094.3,6.7,76.5
13-1199,"Business operations specialists, all other",1157.6,3.9,98.7


  stg_bls__ep_occupation_projections: 1,112 rows x 14 cols, key ['soc_code'] unique | local data/interim/staging/stg_bls__ep_occupation_projections.parquet | DB: 1112


## 5. Summary

In [6]:
display(pd.concat(summary, ignore_index=True))
if conn is not None:
    conn.close()

,table,rows,cols,key,source_key,notebook,in_db,loaded_utc
0,stg_bls__ep_entry_requirements,831,11,soc_code,bls_ep_education,22-load-bls-education.ipynb,831,2026-10-14T21:12:09+00:00
1,stg_bls__ep_attainment,5824,9,soc_code + attainment_level,bls_ep_education,22-load-bls-education.ipynb,5824,2026-10-14T21:12:09+00:00
2,stg_bls__ep_occupation_projections,1112,14,soc_code,bls_ep_occupation,22-load-bls-education.ipynb,1112,2026-10-14T21:12:09+00:00
